In [6]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('scores/S1/perceived_movie/wr_speech/presto.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['window_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('presto', 'WER'): array([ 0.78041856,  0.71696973,  1.14015658,  0.97382558,  0.73502576,
        0.53220014,  0.65505166,  1.35928737,  1.62896459,  1.26640715,
        0.78472857,  0.57945633, -0.1839418 , -0.26280061,  0.37965046,
        0.23606113,  0.08809253, -0.05290915,  0.56976017,  0.13099088,
        0.2305734 ,  0.3309984 , -0.39310793, -0.3762982 , -0.37410636,
       -0.34657144, -0.30032905, -0.92213889, -0.15972417, -0.16793485,
        0.34742403, -0.06106147, -0.42583648,  0.42768588,  0.35706231,
       -0.41555505, -0.50866316, -0.23735633, -0.24426598, -0.20469574,
       -0.5479715 , -1.45984032, -1.67808988, -1.44820369, -0.9822398 ,
       -0.3148335 , -0.92591938, -0.2078275 , -0.58171004,  0.24722569,
        1.02070286,  1.64316767,  0.99559499,  0.6829508 ,  1.05487982,
        0.59735964,  1.47782133,  2.36904369,  2.42258297,  2.34585751,
        2.3953223 ,  3.49176286,  3.29725923,  

In [7]:
window_zscores = {'subject': [], 'wrmodel': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for subject in [1,2,3]:
    for task in ['presto','partlycloudy','laluna']:
        for model in ['speech', 'auditory']:
            scores = np.load(f'scores/S{subject}/perceived_movie/wr_{model}/{task}.npz', allow_pickle=True)['window_zscores'].item()
            # print(scores['window_zscores'].item())
            window_zscores['subject'].append(subject)
            window_zscores['WER'].append(scores[(task, 'WER')])
            window_zscores['BLEU'].append(scores[(task, 'BLEU')])
            window_zscores['METEOR'].append(scores[(task, 'METEOR')])
            window_zscores['BERT'].append(scores[(task, 'BERT')])
            window_zscores['wrmodel'].append(model)

window_zscores

{'subject': [1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3],
 'wrmodel': ['speech',
  'auditory',
  'speech',
  'auditory',
  'speech',
  'auditory',
  'speech',
  'auditory',
  'speech',
  'auditory',
  'speech',
  'auditory',
  'speech',
  'auditory',
  'speech',
  'auditory',
  'speech',
  'auditory'],
 'WER': [array([ 0.78041856,  0.71696973,  1.14015658,  0.97382558,  0.73502576,
          0.53220014,  0.65505166,  1.35928737,  1.62896459,  1.26640715,
          0.78472857,  0.57945633, -0.1839418 , -0.26280061,  0.37965046,
          0.23606113,  0.08809253, -0.05290915,  0.56976017,  0.13099088,
          0.2305734 ,  0.3309984 , -0.39310793, -0.3762982 , -0.37410636,
         -0.34657144, -0.30032905, -0.92213889, -0.15972417, -0.16793485,
          0.34742403, -0.06106147, -0.42583648,  0.42768588,  0.35706231,
         -0.41555505, -0.50866316, -0.23735633, -0.24426598, -0.20469574,
         -0.5479715 , -1.45984032, -1.67808988, -1.44820369, -0.9822398 ,
         -0.3

In [8]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(235+263+331)
m=829

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(9):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

# S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
# S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
# S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
# print(S1,S2,S3,'BERT')

results_df

235
235
263
=
829


,subject,wrmodel,WER,BLEU,METEOR,BERT
0,1,speech,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,1,auditory,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, ..."
2,1,speech,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,1,auditory,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,1,speech,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
5,1,auditory,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
6,2,speech,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
7,2,auditory,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
8,2,speech,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
9,2,auditory,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."


In [9]:
rows = []
for subject in [1, 2, 3]:
    for wrmodel in ['speech', 'auditory']:
        mask = (results_df['subject'] == subject) & (results_df['wrmodel'] == wrmodel)
        subset = results_df[mask]
        bert_values = np.concatenate(subset['BERT'].values).mean()
        rows.append({'subject': subject, 'wrmodel': wrmodel, 'significantly_decoded': bert_values})

to_file = pd.DataFrame(rows)
to_file.to_csv('perceived_movie_percentages.csv', index=False)

to_file

,subject,wrmodel,significantly_decoded
0,1,speech,0.131484
1,1,auditory,0.092883
2,2,speech,0.247286
3,2,auditory,0.376357
4,3,speech,0.290712
5,3,auditory,0.314837


In [10]:
# results = np.load('results/S1/perceived_speech/wheretheressmoke.npz', allow_pickle=True)
# result_names = results.files
# result_files={}
# for name in result_names:
#     result_files[name] = results[name]
# result_files